# Part A :- Concept Appliction

## Import Libraries

In [1]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import numpy as np

## Load and preprocess dataset

In [2]:
def load_and_prepare_data(test_size=0.2, random_state=42):
    """Load the digits dataset, split it into training and testing sets, and scale the features."""
    digits = load_digits()
    X = digits.data
    y = digits.target

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    return X_train_scaled, X_test_scaled, y_train, y_test


## Train SVM with GridSearchCV

In [3]:
def train_svm(X_train, y_train):
    """ Train an SVM classifier using GridSearchCV to find the best hyperparameters."""
    param_grid = {
        'C': [1, 10, 100],
        'gamma': [0.001, 0.01, 0.1]
    }

    grid = GridSearchCV(
        SVC(kernel='rbf'),
        param_grid,
        cv=5,
        scoring='accuracy',
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    print("Best SVM Parameters:", grid.best_params_)
    return grid.best_estimator_

## Train KNN

In [4]:
def train_knn(X_train, y_train, k=3):
    """ Train a KNN classifier with the specified number of neighbors."""
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    return knn

## Evaluate model

In [ ]:
def evaluate_model(model, X_test, y_test, model_name="Model"):
    """ Evaluate the model's performance on the test set and print the results."""
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)

    print(f"\n{model_name} Accuracy: {acc * 100:.4f} \n")
    print(f"{model_name} Confusion Matrix:\n{cm}")
    print(f"{model_name} Classification Report:\n")
    print(classification_report(y_test, y_pred))

    return y_pred, cm


## Find most confused digit pairs

In [ ]:
def find_confused_pairs(cm):
    """ Analyze the confusion matrix to find the most confused digit pairs."""
    cm_copy = cm.copy()
    np.fill_diagonal(cm_copy, 0)

    max_confusion = np.max(cm_copy)
    pairs = np.argwhere(cm_copy == max_confusion)

    print("\nMost confused digit pairs:")
    for pair in pairs:
        print(f"{pair[0]} vs {pair[1]} ({max_confusion} mistakes)")

In [7]:
X_train, X_test, y_train, y_test = load_and_prepare_data()

In [8]:
knn_model = train_knn(X_train, y_train)

knn_pred, knn_cm = evaluate_model(knn_model, X_test, y_test, "KNN")

find_confused_pairs(knn_cm)


KNN Accuracy: 96.6667 

KNN Confusion Matrix:
[[36  0  0  0  0  0  0  0  0  0]
 [ 0 35  1  0  0  0  0  0  0  0]
 [ 0  0 35  0  0  0  0  0  0  0]
 [ 0  0  1 36  0  0  0  0  0  0]
 [ 0  0  0  0 34  0  0  2  0  0]
 [ 0  0  0  0  0 37  0  0  0  0]
 [ 0  0  0  0  0  0 36  0  0  0]
 [ 0  0  1  0  0  0  0 35  0  0]
 [ 0  3  1  0  0  0  0  0 31  0]
 [ 0  0  0  0  1  0  1  0  1 33]]
KNN Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        36
           1       0.92      0.97      0.95        36
           2       0.90      1.00      0.95        35
           3       1.00      0.97      0.99        37
           4       0.97      0.94      0.96        36
           5       1.00      1.00      1.00        37
           6       0.97      1.00      0.99        36
           7       0.95      0.97      0.96        36
           8       0.97      0.89      0.93        35
           9       1.00      0.92      0.96        36

In [9]:
svm_model = train_svm(X_train, y_train)

svm_pred, svm_cm = evaluate_model(svm_model, X_test, y_test, "SVM")

find_confused_pairs(svm_cm)

Best SVM Parameters: {'C': 100, 'gamma': 0.01}

SVM Accuracy: 98.3333 

SVM Confusion Matrix:
[[36  0  0  0  0  0  0  0  0  0]
 [ 0 35  0  0  1  0  0  0  0  0]
 [ 0  0 35  0  0  0  0  0  0  0]
 [ 0  0  0 37  0  0  0  0  0  0]
 [ 0  0  0  0 35  0  0  1  0  0]
 [ 0  0  0  0  0 37  0  0  0  0]
 [ 0  0  0  0  0  0 36  0  0  0]
 [ 0  0  0  0  0  0  0 36  0  0]
 [ 0  1  0  0  1  0  0  0 33  0]
 [ 0  0  0  0  0  0  1  1  0 34]]
SVM Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        36
           1       0.97      0.97      0.97        36
           2       1.00      1.00      1.00        35
           3       1.00      1.00      1.00        37
           4       0.95      0.97      0.96        36
           5       1.00      1.00      1.00        37
           6       0.97      1.00      0.99        36
           7       0.95      1.00      0.97        36
           8       1.00      0.94      0.97        35
      

# Part B :- Stretch Problem

In [10]:
%pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 22.8 MB/s eta 0:00:00


In [11]:
import faiss
import time

## Load Data

In [ ]:
def load_data():
    """ Load the digits dataset and prepare it for training and testing."""
    digits = load_digits()
    X = digits.data.astype('float32')
    y = digits.target

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train).astype('float32')
    X_test = scaler.transform(X_test).astype('float32')

    return X_train, X_test, y_train, y_test

## sklearn KNN timing

In [ ]:
def sklearn_knn_speed(X_train, y_train, X_test):
    """ Measure the prediction time of sklearn's KNN on the first 1000 test samples."""
    knn = KNeighborsClassifier(n_neighbors=3)
    knn.fit(X_train, y_train)

    start = time.time()
    print("Predicting with sklearn KNN...")
    knn.predict(X_test[:1000])
    end = time.time()

    return end - start

##  FAISS KNN timing

In [ ]:

def build_faiss_index(X_train):
    """ Build a FAISS index for the training data."""
    dim = X_train.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(X_train)
    return index


def faiss_knn_speed(index, X_test, k=3):
    """ Measure the prediction time of FAISS KNN on the first 1000 test samples."""
    start = time.time()
    print("Predicting with FAISS KNN...")
    distances, indices = index.search(X_test[:1000], k)
    end = time.time()

    return end - start

In [ ]:
X_train, X_test, y_train, y_test = load_data()

sklearn_time = sklearn_knn_speed(X_train, y_train, X_test)
print(f"sklearn KNN time: {sklearn_time:.6f} sec")

index = build_faiss_index(X_train)
faiss_time = faiss_knn_speed(index, X_test)
print(f"FAISS time: {faiss_time:.6f} sec")

Predicting with sklearn KNN...
sklearn KNN time: 0.006618 sec
Predicting with FAISS KNN...
FAISS time: 0.003293 sec


In [16]:
%pip uninstall faiss-cpu -y

Found existing installation: faiss-cpu 1.13.2
Uninstalling faiss-cpu-1.13.2:
  Successfully uninstalled faiss-cpu-1.13.2



### Findings:

The prediction time of FAISS KNN is significantly faster than that of sklearn's KNN.
On the first 1000 test samples, FAISS KNN takes approximately 0.003293 seconds, while sklearn's KNN takes approximately 0.006618 seconds.
This suggests that FAISS KNN can be a good choice for applications where speed is a concern.



### Why FAISS is faster ?

The main difference between the two algorithms is how they store and query the data.

Sklearn's KNN stores the data in a BallTree data structure, which is a hierarchical data structure that allows for efficient nearest neighbor searches. The algorithm first builds a BallTree from the training data, and then uses this data structure to query the k-nearest neighbors for a given test sample.

FAISS KNN, on the other hand, stores the data in a flat array, and uses the FAISS library to perform the nearest neighbor searches. FAISS is a library that provides a fast and efficient implementation of the k-nearest neighbors algorithm.

In terms of complexity, both algorithms have a time complexity of O(n log n) for building the data structure, and O(log n) for querying the k-nearest neighbors. However, the constant factors for FAISS KNN are much smaller than those for sklearn's KNN, which is why FAISS KNN is generally faster.

